# 02 · Resumen general

Análisis exploratorio y estadística descriptiva de las series.

In [1]:
import sys
sys.path.append("..")

from src import utils

En este notebook se realiza una primera exploración del dataset filtrado de Spotify Charts. El objetivo es entender qué información contiene la base, qué período cubre, qué regiones incluye y qué canciones aparecen con mayor continuidad en el chart.

Este paso sirve como puente entre la carga de datos del notebook `01_carga_y_datos` y los análisis específicos de los notebooks siguientes.

In [2]:
import pandas as pd
import matplotlib.pyplot as plt

RUTA = "../data/charts_ar_global.csv"

df = pd.read_csv(RUTA, parse_dates=["date"])
df.head()

,title,rank,date,artist,url,region,chart,trend,streams
0,Chantaje (feat. Maluma),1,2017-01-01,Shakira,https://open.spotify.com/track/6mICuAdrwEjh6Y6...,Argentina,top200,SAME_POSITION,253019.0
1,I Want You Back,142,2017-01-01,The Jackson 5,https://open.spotify.com/track/3b0EOvScbZUc0qJ...,Global,top200,MOVE_UP,436184.0
2,Thinking out Loud,141,2017-01-01,Ed Sheeran,https://open.spotify.com/track/1Slwb6dOYkBlWal...,Global,top200,SAME_POSITION,436476.0
3,Lighthouse - Andrelli Remix,140,2017-01-01,Hearts & Colors,https://open.spotify.com/track/04CttTezSnv71US...,Global,top200,MOVE_DOWN,439245.0
4,Me llamas (feat. Maluma) - Remix,139,2017-01-01,Piso 21,https://open.spotify.com/track/5hEM0JchdVzQ5Pw...,Global,top200,MOVE_UP,439910.0


Primero observamos la cantidad de filas y columnas, los nombres de las variables y el tipo de dato de cada columna. Esto nos permite verificar que la fecha esté correctamente interpretada como variable temporal y que las columnas necesarias para construir las series estén disponibles.

In [3]:
df.shape

(907101, 9)

In [4]:
df.columns

Index(['title', 'rank', 'date', 'artist', 'url', 'region', 'chart', 'trend',
       'streams'],
      dtype='object')

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 907101 entries, 0 to 907100
Data columns (total 9 columns):
 #   Column   Non-Null Count   Dtype         
---  ------   --------------   -----         
 0   title    907101 non-null  object        
 1   rank     907101 non-null  int64         
 2   date     907101 non-null  datetime64[ns]
 3   artist   907101 non-null  object        
 4   url      907101 non-null  object        
 5   region   907101 non-null  object        
 6   chart    907101 non-null  object        
 7   trend    907101 non-null  object        
 8   streams  726567 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 62.3+ MB


A partir de la estructura observada, la variable `date` ya se encuentra en formato temporal y permite ordenar las observaciones día a día. La variable `streams` será la magnitud principal para construir las series, aunque presenta valores faltantes. Esto se debe a que no todos los tipos de chart reportan streams, por lo que para los análisis basados en cantidad de reproducciones se priorizará el chart `top200`.

In [6]:
df["chart"].value_counts(dropna=False)

chart
top200     726567
viral50    180534
Name: count, dtype: int64

In [7]:
df["region"].value_counts(dropna=False)

region
Argentina    455308
Global       451793
Name: count, dtype: int64

In [8]:
df["date"].min(), df["date"].max()

(Timestamp('2017-01-01 00:00:00'), Timestamp('2021-12-31 00:00:00'))

El dataset filtrado cubre el período 2017-2021, con observaciones diarias para Argentina y Global. Como la frecuencia temporal de los datos es diaria, las series construidas a partir de este dataset se interpretarán como señales discretas con una frecuencia de muestreo de 1 muestra por día.

In [9]:
df_top200 = df[df["chart"] == "top200"].copy()

df_top200.shape

(726567, 9)

In [10]:
df_top50 = df_top200[df_top200["rank"] <= 50].copy()

df_top50.shape

(181627, 9)

A partir del chart `top200`, también se define un subconjunto `top50`, compuesto por las observaciones con `rank <= 50`. Este recorte permite concentrarse en las canciones de mayor popularidad, sin perder la variable `streams`, que será central para construir las señales temporales.

In [11]:
df_top50[["title", "artist", "rank", "date", "region", "streams"]].head()

,title,artist,rank,date,region,streams
0,Chantaje (feat. Maluma),Shakira,1,2017-01-01,Argentina,253019.0
251,Hear Me Now,"Alok, Bruno Martini, Zeeba",49,2017-01-01,Global,873494.0
266,Vente Pa' Ca (feat. Maluma),Ricky Martin,2,2017-01-01,Argentina,223988.0
267,Reggaetón Lento (Bailemos),CNCO,3,2017-01-01,Argentina,210943.0
268,Safari,"J Balvin, Pharrell Williams, BIA, Sky",4,2017-01-01,Argentina,173865.0


Para el análisis del ciclo de vida de una canción, conviene elegir temas que tengan una permanencia suficiente en el chart. Por eso, se agrupan las observaciones por canción y artista, calculando la cantidad de días en el Top 50, el total de streams, el promedio de streams y el mejor ranking alcanzado.

In [12]:
canciones_candidatas = (
    df_top50
    .groupby(["title", "artist"])
    .agg(
        dias_en_top50=("date", "nunique"),
        streams_totales=("streams", "sum"),
        streams_promedio=("streams", "mean"),
        mejor_rank=("rank", "min"),
        peor_rank=("rank", "max"),
        primera_fecha=("date", "min"),
        ultima_fecha=("date", "max"),
        regiones=("region", "nunique")
    )
    .sort_values("dias_en_top50", ascending=False)
)

canciones_candidatas.head(20)

,,dias_en_top50,streams_totales,streams_promedio,mejor_rank,peor_rank,primera_fecha,ultima_fecha,regiones
title,artist,,,,,,,,
Blinding Lights,The Weeknd,749,2.642946e+09,2.524303e+06,1,50,2019-11-29,2021-12-20,2
Someone You Loved,Lewis Capaldi,744,1.704816e+09,2.291419e+06,4,50,2019-03-19,2021-05-20,1
Dance Monkey,Tones And I,559,2.043632e+09,2.544996e+06,1,50,2019-07-31,2021-03-04,2
Watermelon Sugar,Harry Styles,540,1.278839e+09,1.791092e+06,4,50,2019-11-17,2021-11-04,2
Shape of You,Ed Sheeran,511,1.771596e+09,1.934056e+06,1,50,2017-01-06,2019-01-01,2
Sunflower - Spider-Man: Into the Spider-Verse,"Post Malone, Swae Lee",492,1.308800e+09,2.660163e+06,1,50,2018-10-19,2020-03-19,1
Don't Start Now,Dua Lipa,478,1.351617e+09,1.859171e+06,2,50,2019-11-01,2021-06-24,2
bad guy,Billie Eilish,431,1.382444e+09,2.433881e+06,1,50,2019-03-29,2020-06-11,2
Me Rehúso,Danny Ocean,409,5.109219e+08,7.491524e+05,1,50,2017-02-26,2018-04-22,2


In [13]:
df_ar_top50 = df_top50[df_top50["region"] == "Argentina"].copy()

candidatas_ar = (
    df_ar_top50
    .groupby(["title", "artist"])
    .agg(
        dias_en_top50=("date", "nunique"),
        streams_totales=("streams", "sum"),
        streams_promedio=("streams", "mean"),
        mejor_rank=("rank", "min"),
        peor_rank=("rank", "max"),
        primera_fecha=("date", "min"),
        ultima_fecha=("date", "max")
    )
    .sort_values("dias_en_top50", ascending=False)
)

candidatas_ar.head(20)

,,dias_en_top50,streams_totales,streams_promedio,mejor_rank,peor_rank,primera_fecha,ultima_fecha
title,artist,,,,,,,
Me Rehúso,Danny Ocean,409,57107083.0,139626.119804,1,50,2017-02-26,2018-04-22
Shape of You,Ed Sheeran,406,44547066.0,109721.837438,2,50,2017-01-06,2018-02-21
Tusa,"KAROL G, Nicki Minaj",363,70968068.0,195504.319559,1,50,2019-11-19,2021-01-10
Ya No Más,"Fer Palacio, DJ Alex, Santiago Saez",356,32417086.0,91059.230337,18,50,2020-03-20,2021-03-14
Otra vez (feat. J Balvin),Zion & Lennox,341,29548029.0,86651.111437,5,50,2017-01-01,2017-12-10
Adan y Eva,Paulo Londra,340,67878586.0,199642.900000,1,50,2018-11-06,2019-12-15
Despacito (Featuring Daddy Yankee),Luis Fonsi,340,52249954.0,153676.335294,1,50,2017-01-14,2018-01-01
Me llamas (feat. Maluma) - Remix,Piso 21,321,26986037.0,84068.651090,8,49,2017-01-01,2017-11-20
La Rompe Corazones,"Daddy Yankee, Ozuna",319,29668035.0,93003.244514,5,50,2017-01-30,2017-12-17


A partir de las canciones candidatas en Argentina, se elige trabajar inicialmente con **“Tusa” — KAROL G, Nicki Minaj**. Esta canción tiene una permanencia prolongada en el Top 50 argentino, alcanza el puesto 1 y aparece dentro del período cubierto por el dataset, lo que permite observar mejor su ciclo de vida dentro del chart: entrada, crecimiento, pico y caída.

In [14]:
CANCION = "Tusa"
ARTISTA = "KAROL G, Nicki Minaj"

cancion_elegida = df_ar_top50[
    (df_ar_top50["title"] == CANCION) &
    (df_ar_top50["artist"] == ARTISTA)
].sort_values("date")

cancion_elegida.head()

,title,rank,date,artist,url,region,chart,trend,streams
523226,Tusa,43,2019-11-19,"KAROL G, Nicki Minaj",https://open.spotify.com/track/7k4t7uLgtOxPwTp...,Argentina,top200,MOVE_UP,73707.0
523435,Tusa,32,2019-11-20,"KAROL G, Nicki Minaj",https://open.spotify.com/track/7k4t7uLgtOxPwTp...,Argentina,top200,MOVE_UP,86089.0
523816,Tusa,24,2019-11-21,"KAROL G, Nicki Minaj",https://open.spotify.com/track/7k4t7uLgtOxPwTp...,Argentina,top200,MOVE_UP,101383.0
524273,Tusa,23,2019-11-22,"KAROL G, Nicki Minaj",https://open.spotify.com/track/7k4t7uLgtOxPwTp...,Argentina,top200,MOVE_UP,123424.0
524771,Tusa,22,2019-11-23,"KAROL G, Nicki Minaj",https://open.spotify.com/track/7k4t7uLgtOxPwTp...,Argentina,top200,MOVE_UP,128233.0


In [15]:
cancion_elegida[["date", "title", "artist", "rank", "streams"]].head()

,date,title,artist,rank,streams
523226,2019-11-19,Tusa,"KAROL G, Nicki Minaj",43,73707.0
523435,2019-11-20,Tusa,"KAROL G, Nicki Minaj",32,86089.0
523816,2019-11-21,Tusa,"KAROL G, Nicki Minaj",24,101383.0
524273,2019-11-22,Tusa,"KAROL G, Nicki Minaj",23,123424.0
524771,2019-11-23,Tusa,"KAROL G, Nicki Minaj",22,128233.0


In [16]:
cancion_elegida[["date", "title", "artist", "rank", "streams"]].tail()

,date,title,artist,rank,streams
730116,2021-01-06,Tusa,"KAROL G, Nicki Minaj",48,72504.0
730422,2021-01-07,Tusa,"KAROL G, Nicki Minaj",47,81250.0
730868,2021-01-08,Tusa,"KAROL G, Nicki Minaj",49,86065.0
731264,2021-01-09,Tusa,"KAROL G, Nicki Minaj",49,93155.0
732133,2021-01-10,Tusa,"KAROL G, Nicki Minaj",48,89769.0


En este notebook se realizó una primera exploración del dataset filtrado de Spotify Charts. Se verificó que la base contiene observaciones diarias entre 2017 y 2021 para Argentina y Global, y que el chart `top200` es el más adecuado para trabajar con `streams`.

Además, se construyó un subconjunto `top50` filtrando las canciones con `rank <= 50`, para concentrar el análisis en canciones de alta popularidad. A partir de la permanencia en el chart argentino, se seleccionó **“Tusa” — KAROL G, Nicki Minaj** como canción inicial para estudiar su ciclo de vida en el notebook `03_ciclo_de_vida`.

En el siguiente notebook, esta canción será tratada como una señal temporal discreta, donde cada muestra diaria representa la cantidad de streams en Argentina.